In [20]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt
from helpful_functions import *
# import pandas
# from tqdm import tqdm

In [21]:
drive_parameters = {
    'N_q': 14,   
    'N_c': 18,
    'detuning': 165*2*np.pi,  # qubit-resonator detuning in angular frequency
    'gbs': 1.0*2*np.pi,       # qubit-resonator coupling strength in angular frequency
}
alpha = -160.0*2*np.pi  # qubit anharmonicity in MHz angular frequency
results = dchi_H(drive_parameters, alpha)
print(f'Dispersive shift chi list: {np.array(results['chi_dict']['chi_list'])/(2*np.pi)} MHz')

Dispersive shift chi list: [-0.36092235 -0.31612066 -0.2841849  -0.25992822 -0.24069082 -0.22494786
 -0.21175365 -0.20048623 -0.19071741 -0.18214149 -0.17453354 -0.16772389
 -0.16158181 -0.1560047  -0.15091079 -0.14623397 -0.2206356 ] MHz


In [22]:
a_r_dressed = results['dressed_operators']['a_r']
a_q_dressed = results['dressed_operators']['a_q']
w_r_dressed = results['dressed_operators']['w_r_dressed']
w_q_dressed = results['dressed_operators']['w_q_dressed']
chi = results['chi_dict']['chi_list'][0]  # dispersive shift for 0->1 transition

H0_approx = w_r_dressed * a_r_dressed.dag() * a_r_dressed + w_q_dressed * a_q_dressed.dag() * a_q_dressed + chi * a_r_dressed.dag() * a_r_dressed * a_q_dressed.dag() * a_q_dressed
H0_exact = results['Hamiltonian_dict']['H0']

evals_approx, evecs_approx = H0_approx.eigenstates()
sorted_dict_approx = sort_eigenvalues_eigenstates_by_excitation_number(evals_approx, evecs_approx, drive_parameters['N_c'], drive_parameters['N_q'], results['Hamiltonian_dict']['a_r'], results['Hamiltonian_dict']['a_q'])
# sorted_dict_approx = sort_eigenvalues_eigenstates_by_excitation_number(evals_approx, evecs_approx, drive_parameters['N_c'], drive_parameters['N_q'], a_r_dressed, a_q_dressed)

evals_sorted_approx = sorted_dict_approx['evals_sorted']
evals_sorted_exact = results['sorted_dict']['evals_sorted']
evecs_sorted_approx = sorted_dict_approx['evecs_sorted']
evecs_sorted_exact = results['sorted_dict']['evecs_sorted']

Nr_subspace_dim = 2
Nq_subspace_dim = 2

Projector = projector_onto_states([results['sorted_dict']['evecs_sorted'][n_r][n_q] for n_r in range(Nr_subspace_dim) for n_q in range(Nq_subspace_dim)])

H0_approx_nodressed = w_r_dressed * results['Hamiltonian_dict']['a_r'].dag() * results['Hamiltonian_dict']['a_r'] + w_q_dressed * results['Hamiltonian_dict']['a_q'].dag() * results['Hamiltonian_dict']['a_q'] + chi * results['Hamiltonian_dict']['a_r'].dag() * results['Hamiltonian_dict']['a_r'] * results['Hamiltonian_dict']['a_q'].dag() * results['Hamiltonian_dict']['a_q'] # if we don't use dressed operators. this is close but defintely not exact. Exact is with dressed operators.

# H0_approx = w_q_dressed * a_q_dressed.dag() * a_q_dressed + chi * a_r_dressed.dag() * a_r_dressed * a_q_dressed.dag() * a_q_dressed, without resonator term. this term is important for exactness but is small. in the 2 by 2 subspace we get smaller inner product error without it.

print(evecs_sorted_exact[0][1].dag()*evecs_sorted_approx[0][1])
# print(sorted_dict_approx['overrides'])
#print(results['sorted_dict']['overrides'])
# print(Projector.tr())
print(normalized_innerproduct_projected(H0_approx,H0_exact,Projector))
print(normalized_innerproduct_projected(H0_approx_nodressed,H0_exact,Projector))


(0.9999999999999996+0j)
(1.0000000000000002+0j)
(0.9854307214814209+0j)


In [23]:
evals_qobj = results['sorted_dict']['evals_qobj']
evecs_qobj = results['sorted_dict']['evecs_qobj']

print(norm(evecs_qobj.dag()*H0_exact*evecs_qobj - evals_qobj)) # should be zero to show they are the same

9.334149615127226e-10
